In [1]:
pip install pandas numpy scikit-learn catboost torch transformers datasets accelerate codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.4/372.4 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.6 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5


In [13]:
import os
import time
import json
import numpy as np
import pandas as pd
import torch

from catboost import CatBoostRegressor, Pool
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score

# =========================================================
# CONFIG
# =========================================================
TEST_FILE = "/content/jira_TD_dataset_clean.csv"
CATBOOST_MODEL_FILE = "/content/catboost_td_distilled.cbm"
CATBOOST_META_FILE = "/content/catboost_td_distilled_meta.json"
TRANSFORMER_MODEL_NAME = "karths/binary_classification_train_TD"

TEXT_COL = "testo"
LABEL_COL = "label"
N_SAMPLES = 300
MAX_LENGTH = 256

# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv(TEST_FILE)
df.columns = [c.strip() for c in df.columns]
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df[df[LABEL_COL].isin([0, 1])].copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

df = df.head(N_SAMPLES).reset_index(drop=True)

texts = df[TEXT_COL].tolist()
y_true = df[LABEL_COL].values

print(f"Frasi usate: {len(df)}")

# =========================================================
# CATBOOST
# =========================================================
print("\n" + "="*70)
print("CATBOOST")
print("="*70)

cat_model = CatBoostRegressor()
cat_model.load_model(CATBOOST_MODEL_FILE)

threshold = 0.5
if os.path.exists(CATBOOST_META_FILE):
    with open(CATBOOST_META_FILE, "r", encoding="utf-8") as f:
        meta = json.load(f)
        threshold = meta.get("best_threshold", 0.5)

# compatibilità col modello
df_cat = df.rename(columns={TEXT_COL: "orig_text"}).copy()

cat_times = []
cat_probs = []

cat_device = "CPU"

for text in df_cat["orig_text"]:
    pool = Pool(
        data=pd.DataFrame({"orig_text": [text]}),
        text_features=["orig_text"]
    )

    start = time.perf_counter()
    pred_logit = cat_model.predict(pool, task_type=cat_device)
    end = time.perf_counter()

    pred_prob = 1.0 / (1.0 + np.exp(-pred_logit[0]))
    cat_probs.append(pred_prob)
    cat_times.append((end - start) * 1000)  # ms

cat_preds = (np.array(cat_probs) >= threshold).astype(int)
cat_acc = accuracy_score(y_true, cat_preds)

print(f"Device: {cat_device}")
print(f"Accuracy: {cat_acc:.4f}")
print(f"Tempo medio per frase: {np.mean(cat_times):.4f} ms")
print(f"Tempo min per frase:   {np.min(cat_times):.4f} ms")
print(f"Tempo max per frase:   {np.max(cat_times):.4f} ms")

# =========================================================
# TRANSFORMER
# =========================================================
print("\n" + "="*70)
print("TRANSFORMER")
print("="*70)

tr_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)
tr_model = AutoModelForSequenceClassification.from_pretrained(TRANSFORMER_MODEL_NAME)
tr_model.to(tr_device)
tr_model.eval()

tr_times = []
tr_probs = []

with torch.no_grad():
    for text in texts:
        inputs = tokenizer(
            [text],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(tr_device)

        if tr_device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()

        outputs = tr_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

        if tr_device.type == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()

        tr_probs.append(probs[0, 1].item())
        tr_times.append((end - start) * 1000)  # ms

tr_preds = (np.array(tr_probs) >= 0.5).astype(int)
tr_acc = accuracy_score(y_true, tr_preds)

print(f"Device: {'GPU' if tr_device.type == 'cuda' else 'CPU'}")
print(f"Accuracy: {tr_acc:.4f}")
print(f"Tempo medio per frase: {np.mean(tr_times):.4f} ms")
print(f"Tempo min per frase:   {np.min(tr_times):.4f} ms")
print(f"Tempo max per frase:   {np.max(tr_times):.4f} ms")

# =========================================================
# CONFRONTO FINALE
# =========================================================
print("\n" + "="*70)
print("CONFRONTO FINALE")
print("="*70)

print(f"CatBoost   | Device: {cat_device:>3} | Accuracy: {cat_acc:.4f} | "
      f"Mean: {np.mean(cat_times):.4f} ms | Min: {np.min(cat_times):.4f} ms | Max: {np.max(cat_times):.4f} ms")

print(f"Transformer| Device: {'GPU' if tr_device.type == 'cuda' else 'CPU':>3} | Accuracy: {tr_acc:.4f} | "
      f"Mean: {np.mean(tr_times):.4f} ms | Min: {np.min(tr_times):.4f} ms | Max: {np.max(tr_times):.4f} ms")

Frasi usate: 300

CATBOOST
Device: CPU
Accuracy: 0.7233
Tempo medio per frase: 1.1483 ms
Tempo min per frase:   0.7523 ms
Tempo max per frase:   3.1739 ms

TRANSFORMER


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Device: GPU
Accuracy: 0.7033
Tempo medio per frase: 6.0421 ms
Tempo min per frase:   4.1072 ms
Tempo max per frase:   27.4520 ms

CONFRONTO FINALE
CatBoost   | Device: CPU | Accuracy: 0.7233 | Mean: 1.1483 ms | Min: 0.7523 ms | Max: 3.1739 ms
Transformer| Device: GPU | Accuracy: 0.7033 | Mean: 6.0421 ms | Min: 4.1072 ms | Max: 27.4520 ms


In [7]:
import os
import json
import time
import warnings
from typing import Optional, Tuple, Dict

import numpy as np
import pandas as pd
import torch

from catboost import CatBoostRegressor, Pool
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# =========================================================
# CONFIG
# =========================================================
TEST_FILE = "/content/jira_TD_dataset_clean.csv"
MODEL_FILE = "/content/catboost_td_distilled.cbm"
META_FILE = "/content/catboost_td_distilled_meta.json"
TRANSFORMER_MODEL = "karths/binary_classification_train_TD"
RESULTS_DIR = "benchmark_inference_results"

TEXT_COL = "testo"
LABEL_COL = "label"
TRANSFORMER_BATCH_SIZE = 32
TRANSFORMER_MAX_LENGTH = 256

os.makedirs(RESULTS_DIR, exist_ok=True)
warnings.filterwarnings("ignore")


# =========================================================
# Utility
# =========================================================

def clean_dataframe(df: pd.DataFrame, text_col: str, label_col: str) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip() for c in df.columns]

    if text_col not in df.columns:
        raise ValueError(f"Colonna testo '{text_col}' non trovata. Colonne disponibili: {df.columns.tolist()}")

    if label_col not in df.columns:
        raise ValueError(f"Colonna label '{label_col}' non trovata. Colonne disponibili: {df.columns.tolist()}")

    df[text_col] = df[text_col].astype(str).fillna("").str.strip()
    df = df[df[text_col].str.len() > 0].copy()

    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df[df[label_col].isin([0, 1])].copy()
    df[label_col] = df[label_col].astype(int)

    if len(df) == 0:
        raise ValueError("Dataset vuoto dopo la pulizia.")

    return df.reset_index(drop=True)


def get_hw_info() -> Dict:
    info = {
        "cpu_count": os.cpu_count(),
        "torch_cuda_available": torch.cuda.is_available(),
        "torch_cuda_version": torch.version.cuda,
        "gpu_name": None,
        "gpu_count": 0
    }
    if torch.cuda.is_available():
        info["gpu_count"] = torch.cuda.device_count()
        info["gpu_name"] = torch.cuda.get_device_name(0)
    return info


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(y_true: np.ndarray,
                    y_prob: np.ndarray,
                    threshold: float = 0.5) -> Dict:
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else None,
        "pr_auc": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else None,
        "confusion_matrix": cm.tolist(),
        "true_negatives": int(cm[0, 0]),
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "true_positives": int(cm[1, 1]),
    }


# =========================================================
# CATBOOST BENCHMARK
# =========================================================

def benchmark_catboost(df_test: pd.DataFrame,
                       text_col: str,
                       label_col: str,
                       model_file: str,
                       meta_file: Optional[str] = None,
                       device: str = "CPU"):
    print("\n" + "=" * 80)
    print(f"CATBOOST - INFERENCE {device}")
    print("=" * 80)

    # Il tuo modello è salvato come regressor
    model = CatBoostRegressor()
    model.load_model(model_file)

    training_meta = {}
    if meta_file and os.path.exists(meta_file):
        with open(meta_file, "r", encoding="utf-8") as f:
            training_meta = json.load(f)

    threshold = training_meta.get("best_threshold", 0.5)

    # Compatibilità con il modello: rinomina text -> orig_text
    df_eval = df_test.copy()
    df_eval = df_eval.rename(columns={text_col: "orig_text"})
    eval_text_col = "orig_text"

    test_pool = Pool(
        data=df_eval[[eval_text_col]],
        text_features=[eval_text_col]
    )

    y_true = df_eval[label_col].astype(int).values

    # Warmup
    _ = model.predict(test_pool, task_type=device)

    start_time = time.time()
    pred_logit = model.predict(test_pool, task_type=device)
    inference_time = time.time() - start_time

    pred_logit = np.asarray(pred_logit)
    pred_prob = sigmoid(pred_logit)

    metrics = compute_metrics(y_true, pred_prob, threshold=threshold)
    n = len(df_eval)

    result = {
        "model_name": "CatBoost Distilled",
        "model_file": model_file,
        "device": device,
        "test_samples": int(n),
        "inference_time_seconds": float(inference_time),
        "inference_time_ms_per_sample": float((inference_time / n) * 1000.0),
        "throughput_samples_per_sec": float(n / inference_time),
        **metrics
    }

    pred_df = df_eval.copy()
    pred_df["pred_logit"] = pred_logit
    pred_df["pred_prob"] = pred_prob
    pred_df["pred_label"] = (pred_prob >= threshold).astype(int)

    print(json.dumps(result, indent=2))

    return result, pred_df


# =========================================================
# TRANSFORMER BENCHMARK
# =========================================================

def benchmark_transformer(df_test: pd.DataFrame,
                          text_col: str,
                          label_col: str,
                          model_name: str,
                          device: str = "cpu",
                          batch_size: int = 32,
                          max_length: int = 256,
                          threshold: float = 0.5):
    print("\n" + "=" * 80)
    print(f"TRANSFORMER - INFERENCE {'GPU' if device == 'cuda' else 'CPU'}")
    print("=" * 80)

    device_t = torch.device(device)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device_t)
    model.eval()

    texts = df_test[text_col].astype(str).tolist()
    y_true = df_test[label_col].astype(int).values

    # Warmup
    warmup_texts = texts[:min(batch_size, len(texts))]
    if warmup_texts:
        with torch.no_grad():
            inputs = tokenizer(
                warmup_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device_t)
            _ = model(**inputs)

    all_probs = []

    start_time = time.time()

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device_t)

            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)
            batch_probs = probs[:, 1].detach().cpu().numpy()

            all_probs.extend(batch_probs.tolist())

    if device == "cuda":
        torch.cuda.synchronize()

    inference_time = time.time() - start_time

    pred_prob = np.array(all_probs)
    metrics = compute_metrics(y_true, pred_prob, threshold=threshold)
    n = len(df_test)

    result = {
        "model_name": "Transformer",
        "transformer_model": model_name,
        "device": "GPU" if device == "cuda" else "CPU",
        "batch_size": int(batch_size),
        "max_length": int(max_length),
        "test_samples": int(n),
        "inference_time_seconds": float(inference_time),
        "inference_time_ms_per_sample": float((inference_time / n) * 1000.0),
        "throughput_samples_per_sec": float(n / inference_time),
        **metrics
    }

    pred_df = df_test.copy()
    pred_df["pred_prob"] = pred_prob
    pred_df["pred_label"] = (pred_prob >= threshold).astype(int)

    print(json.dumps(result, indent=2))

    return result, pred_df


# =========================================================
# MAIN
# =========================================================

print("=" * 90)
print("BENCHMARK INFERENCE ONLY - CATBOOST vs TRANSFORMER")
print("=" * 90)

# Dataset
df = pd.read_csv(TEST_FILE)
df = clean_dataframe(df, TEXT_COL, LABEL_COL)

print(f"\nDataset: {TEST_FILE}")
print(f"Text column : {TEXT_COL}")
print(f"Label column: {LABEL_COL}")
print(f"Samples test: {len(df)}")
print(f"Distribuzione label:\n{df[LABEL_COL].value_counts().sort_index()}")

hw = get_hw_info()
print("\nHardware rilevato:")
print(json.dumps(hw, indent=2))

summary = {
    "hardware": hw,
    "dataset": {
        "csv": TEST_FILE,
        "text_col": TEXT_COL,
        "label_col": LABEL_COL,
        "test_size": len(df)
    },
    "models": {}
}

comparison_rows = []

# ---------------------------------------------------------
# CatBoost CPU
# ---------------------------------------------------------
cat_cpu_result, cat_cpu_pred = benchmark_catboost(
    df_test=df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    model_file=MODEL_FILE,
    meta_file=META_FILE,
    device="CPU"
)
summary["models"]["catboost"] = {"inference_cpu": cat_cpu_result}
cat_cpu_pred.to_csv(os.path.join(RESULTS_DIR, "predictions_catboost_cpu.csv"), index=False)
comparison_rows.append(cat_cpu_result)

# ---------------------------------------------------------
# CatBoost GPU
# ---------------------------------------------------------
if torch.cuda.is_available():
    try:
        cat_gpu_result, cat_gpu_pred = benchmark_catboost(
            df_test=df,
            text_col=TEXT_COL,
            label_col=LABEL_COL,
            model_file=MODEL_FILE,
            meta_file=META_FILE,
            device="GPU"
        )
        summary["models"]["catboost"]["inference_gpu"] = cat_gpu_result
        cat_gpu_pred.to_csv(os.path.join(RESULTS_DIR, "predictions_catboost_gpu.csv"), index=False)
        comparison_rows.append(cat_gpu_result)
    except Exception as e:
        summary["models"]["catboost"]["inference_gpu"] = {"error": str(e)}
        print(f"[CatBoost GPU] inferenza fallita: {e}")

# ---------------------------------------------------------
# Transformer CPU
# ---------------------------------------------------------
tr_cpu_result, tr_cpu_pred = benchmark_transformer(
    df_test=df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    model_name=TRANSFORMER_MODEL,
    device="cpu",
    batch_size=TRANSFORMER_BATCH_SIZE,
    max_length=TRANSFORMER_MAX_LENGTH,
    threshold=0.5
)
summary["models"]["transformer"] = {"inference_cpu": tr_cpu_result}
tr_cpu_pred.to_csv(os.path.join(RESULTS_DIR, "predictions_transformer_cpu.csv"), index=False)
comparison_rows.append(tr_cpu_result)

# ---------------------------------------------------------
# Transformer GPU
# ---------------------------------------------------------
if torch.cuda.is_available():
    tr_gpu_result, tr_gpu_pred = benchmark_transformer(
        df_test=df,
        text_col=TEXT_COL,
        label_col=LABEL_COL,
        model_name=TRANSFORMER_MODEL,
        device="cuda",
        batch_size=TRANSFORMER_BATCH_SIZE,
        max_length=TRANSFORMER_MAX_LENGTH,
        threshold=0.5
    )
    summary["models"]["transformer"]["inference_gpu"] = tr_gpu_result
    tr_gpu_pred.to_csv(os.path.join(RESULTS_DIR, "predictions_transformer_gpu.csv"), index=False)
    comparison_rows.append(tr_gpu_result)

# ---------------------------------------------------------
# Save results
# ---------------------------------------------------------
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(os.path.join(RESULTS_DIR, "comparison.csv"), index=False)

with open(os.path.join(RESULTS_DIR, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 90)
print("CONFRONTO FINALE")
print("=" * 90)

if not comparison_df.empty:
    cols = [
        "model_name", "device", "accuracy", "precision", "recall", "f1_score",
        "roc_auc", "pr_auc", "inference_time_seconds",
        "inference_time_ms_per_sample", "throughput_samples_per_sec"
    ]
    print(
        comparison_df[cols]
        .sort_values(["accuracy", "inference_time_ms_per_sample"], ascending=[False, True])
        .to_string(index=False)
    )

print("\nFile salvati in:", RESULTS_DIR)
print("- summary.json")
print("- comparison.csv")
print("- predictions_catboost_cpu.csv")
if torch.cuda.is_available():
    print("- predictions_catboost_gpu.csv")
print("- predictions_transformer_cpu.csv")
if torch.cuda.is_available():
    print("- predictions_transformer_gpu.csv")

BENCHMARK INFERENCE ONLY - CATBOOST vs TRANSFORMER

Dataset: /content/jira_TD_dataset_clean.csv
Text column : testo
Label column: label
Samples test: 833
Distribuzione label:
label
0    457
1    376
Name: count, dtype: int64

Hardware rilevato:
{
  "cpu_count": 2,
  "torch_cuda_available": true,
  "torch_cuda_version": "12.8",
  "gpu_name": "Tesla T4",
  "gpu_count": 1
}

CATBOOST - INFERENCE CPU
{
  "model_name": "CatBoost Distilled",
  "model_file": "/content/catboost_td_distilled.cbm",
  "device": "CPU",
  "test_samples": 833,
  "inference_time_seconds": 0.0270998477935791,
  "inference_time_ms_per_sample": 0.03253283048448872,
  "throughput_samples_per_sec": 30738.1800202349,
  "threshold": 0.394784236285393,
  "accuracy": 0.6866746698679472,
  "precision": 0.6619718309859155,
  "recall": 0.625,
  "f1_score": 0.6429548563611491,
  "roc_auc": 0.7644094231575027,
  "pr_auc": 0.7283377515337139,
  "confusion_matrix": [
    [
      337,
      120
    ],
    [
      141,
      235
    ]

config.json:   0%|          | 0.00/740 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{
  "model_name": "Transformer",
  "transformer_model": "karths/binary_classification_train_TD",
  "device": "CPU",
  "batch_size": 32,
  "max_length": 256,
  "test_samples": 833,
  "inference_time_seconds": 183.02132558822632,
  "inference_time_ms_per_sample": 219.71347609631013,
  "throughput_samples_per_sec": 4.551382180862023,
  "threshold": 0.5,
  "accuracy": 0.6854741896758704,
  "precision": 0.6524064171122995,
  "recall": 0.648936170212766,
  "f1_score": 0.6506666666666666,
  "roc_auc": 0.7485770985613855,
  "pr_auc": 0.7131694897727735,
  "confusion_matrix": [
    [
      327,
      130
    ],
    [
      132,
      244
    ]
  ],
  "true_negatives": 327,
  "false_positives": 130,
  "false_negatives": 132,
  "true_positives": 244
}

TRANSFORMER - INFERENCE GPU


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{
  "model_name": "Transformer",
  "transformer_model": "karths/binary_classification_train_TD",
  "device": "GPU",
  "batch_size": 32,
  "max_length": 256,
  "test_samples": 833,
  "inference_time_seconds": 5.585859775543213,
  "inference_time_ms_per_sample": 6.705714016258359,
  "throughput_samples_per_sec": 149.12655051728228,
  "threshold": 0.5,
  "accuracy": 0.6854741896758704,
  "precision": 0.6524064171122995,
  "recall": 0.648936170212766,
  "f1_score": 0.6506666666666666,
  "roc_auc": 0.7485770985613855,
  "pr_auc": 0.7131694897727735,
  "confusion_matrix": [
    [
      327,
      130
    ],
    [
      132,
      244
    ]
  ],
  "true_negatives": 327,
  "false_positives": 130,
  "false_negatives": 132,
  "true_positives": 244
}

CONFRONTO FINALE
        model_name device  accuracy  precision   recall  f1_score  roc_auc   pr_auc  inference_time_seconds  inference_time_ms_per_sample  throughput_samples_per_sec
CatBoost Distilled    CPU  0.686675   0.661972 0.625000  0.642955 